# 01 – Explore QB dropbacks

Loads cached play-by-play data (2016–present) and filters it to QB dropbacks. Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [ ]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp

pl.Config.set_tbl_cols(30)

In [ ]:
pbp = load_pbp()
pbp.shape

## Filter to QB dropbacks

`qb_dropback == 1` covers pass attempts, sacks, and QB scrambles.

In [ ]:
dropbacks = pbp.filter(pl.col("qb_dropback") == 1)
dropbacks.shape

## Columns most relevant to rating QBs

| column | meaning |
|---|---|
| `season`, `week`, `season_type` | when the play happened (`REG` / `POST`) |
| `game_id`, `posteam`, `defteam` | game, offense, defense |
| `passer_player_id`, `passer_player_name` | the QB on the dropback |
| `epa` | expected points added by the play (our target is next-week EPA per dropback) |
| `qb_epa` | EPA, but not penalizing the QB for a receiver's fumble after the catch |
| `cpoe` | completion % over expected (pass attempts only; null otherwise) |
| `pass_attempt`, `complete_pass`, `sack`, `qb_scramble`, `interception` | play outcome flags |
| `air_yards`, `yards_after_catch`, `yards_gained` | yardage |
| `success` | 1 if EPA > 0 |
| `down`, `ydstogo`, `wp` | situation: down, distance, win probability |

In [ ]:
qb_cols = [
    "season", "week", "season_type", "game_id", "posteam", "defteam",
    "passer_player_id", "passer_player_name",
    "qb_dropback", "epa", "qb_epa", "cpoe",
    "pass_attempt", "complete_pass", "sack", "qb_scramble", "interception",
    "air_yards", "yards_after_catch", "yards_gained", "success",
    "down", "ydstogo", "wp",
]
qb = dropbacks.select(qb_cols)
qb.head(10)

In [ ]:
qb.schema

# Phase 2: Explore + define the target

**Leakage guard:** every decision about the target is made by looking at the **training seasons only (2016–2023)**.
2024 (val) and 2025 (test) stay unseen, so their scores later are honest.

In [ ]:
EXPLORE_SEASONS = list(range(2016, 2024))  # train only; do not widen this
train_pbp = pbp.filter(pl.col("season").is_in(EXPLORE_SEASONS))
train_pbp.select(pl.col("season").min().alias("first"), pl.col("season").max().alias("last"))

## Concept 1: one row per QB per game

Like turning a box of grocery receipts into one line per shopping trip: sort dropbacks into piles by
(game, team, passer), then write one summary line per pile with `dropbacks` and `epa_per_db`.
The code lives in `src/features/qb_games.py` so later phases reuse the exact same table.

In [ ]:
from src.features.qb_games import qb_games

games = qb_games(train_pbp)

# How many team-games (one offense in one game) vs QB-games?
team_games = games.select(pl.struct("game_id", "posteam").n_unique()).item()
print(f"team-games: {team_games:,}   QB-games: {games.height:,}   ratio: {games.height / team_games:.2f}")

# How many different passers did each team-game have?
(games.group_by("game_id", "posteam").agg(pl.len().alias("passers"))
      .group_by("passers").agg(pl.len().alias("team_games")).sort("passers"))

### ▶ Play: Concept 1
Change a knob, rerun this cell, and compare against the default (`MIN_DROPBACKS = 1`).

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
MIN_DROPBACKS = 1                 # Try 3, then 10, then 20. Who disappears first? QBs or trick-play passers?
SHOW_GAME = "2023_01_DET_KC"      # Any game_id, format "<season>_<week 2 digits>_<AWAY>_<HOME>", e.g. "2022_10_SEA_TB"
# ---------------------------------------------------------------------------

kept = games.filter(pl.col("dropbacks") >= MIN_DROPBACKS)
dropped = games.filter(pl.col("dropbacks") < MIN_DROPBACKS)

print(f"Default (MIN_DROPBACKS=1): {games.height:,} rows")
print(f"Now (MIN_DROPBACKS={MIN_DROPBACKS}): {kept.height:,} rows  ->  dropped {dropped.height:,} "
      f"({dropped.height / games.height:.1%})")

if dropped.height:
    print("\nPassers who lost the most rows:")
    print(dropped.group_by("passer").agg(pl.len().alias("rows_dropped"), pl.col("dropbacks").sum().alias("dropbacks"))
                 .sort("rows_dropped", descending=True).head(10))

print(f"\nGame {SHOW_GAME} (rows marked kept=false would be dropped):")
games.filter(pl.col("game_id") == SHOW_GAME).with_columns(
    (pl.col("dropbacks") >= MIN_DROPBACKS).alias("kept")
).select("posteam", "passer", "dropbacks", "epa_sum", "epa_per_db", "kept")

## Your exploration

Add cells below.